# Notebook 08: Is the decoupling inside cell states, or just a change in the state mix?

**Goal.** Compute MLCI within each state separately and show how state proportions change with age (hypothesis H3).

**Cell 1.** Loads the tools and the shared settings. **sys.path.insert** tells Python where the project code lives; **mlc.config** gives every notebook the same folders (PROC = processed data, RES = results tables, FIG = figures) and the profile (*sample* = built-in simulated data, *real* = your downloads). It also tells pandas to print wide tables on one screen. This cell is identical in every notebook.

In [ ]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from mlc.config import *
from mlc import genesets as G
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 20)
print("profile:", PROFILE, "| cohorts:", cohorts())

**Cell 2.** Splits each mouse's cells by state and computes the MLCI inside each state (state is now the grouping, so it is not adjusted for). States with fewer than 30 cells in a mouse are skipped (**min_cells=30**), so rare states may have too few mice to analyse.

*You should see:* A table: number of mice per state and cohort (homeostatic and DAM have most).

In [ ]:
from mlc.dataset import Dataset
from mlc.coupling import *
from mlc import plots
sheet = pd.read_csv(SHEET).set_index("sample_id")
obsd = {c: Dataset.load(PROC / f"{c}_scored").obs for c in cohorts()["mouse"]}
rows = []
for c, o in obsd.items():
    for st, d in o.groupby("state"):
        r = coupling_per_sample(d, "m_mitophagy", "m_lysosome", covars=["depth", "pct_mt", "dissoc"],
                                min_cells=30, n_boot=0)
        r["state"], r["cohort"] = st, c; rows.append(r)
T = pd.concat(rows, ignore_index=True)
T["age_months"] = T.sample_id.map(sheet.age_months).astype(float); T["log2_age"] = np.log2(T.age_months)
print(T.groupby(["state", "cohort"]).size().unstack())

**Cell 3.** For each state with data from at least two cohorts, pools the age slope across cohorts. **Reading it:** a negative slope with an interval below 0 inside the homeostatic state means decoupling happens within cells of the same type, not only because more cells became DAM.

*You should see:* Homeostatic: slope about -0.06 with an interval clearly below 0. DAM: negative but with a wide interval that includes 0 (few young DAM cells): report that honestly.

In [ ]:
S_state = cohort_slopes(T, by=("state", "cohort"))
res = {st: dersimonian_laird(g.beta, g.se) for st, g in S_state.groupby("state") if len(g) >= 2}
out = pd.DataFrame(res).T[["beta", "ci_lo", "ci_hi", "p", "k"]].round(3)
print(out); out.to_csv(RES / "mlci_state_meta.csv")

**Cell 4.** State composition per mouse (fraction of its cells in each state), averaged by age. This shows what *does* change by composition (DAM rises with age), so you can state honestly that both effects exist.

*You should see:* DAM fraction rising from about 5% to 25-30% with age.

In [ ]:
comp = (pd.concat([o[["sample_id", "state"]] for o in obsd.values()])
        .groupby("sample_id").state.value_counts(normalize=True).unstack(fill_value=0))
comp["age_months"] = comp.index.map(sheet.age_months).astype(float)
comp.to_csv(RES / "state_proportions.csv")
print(comp.groupby("age_months").mean().round(2))

**Cell 5.** **Figure 3.** (a) Mean MLCI inside each state by age: lines decline in every large state. (b) Composition by age: DAM expands. Together: decoupling is a within-state property, in addition to a composition shift.

In [ ]:
plots.fig_state_res(T, comp); plt.show()